# Hallway Illuminance Train/Eval All-in-One

This notebook is the main user interface for the project. It is designed for Google Colab and walks through dependency setup, Google Drive dataset preparation, manifest generation, sample previews, model setup, training scaffolding, evaluation scaffolding, point-wise reporting, carbon estimation, checkpoint saving, ONNX export, and single-image inference.

## 1. Project Overview

The main task is hallway floor-plane illuminance estimation. Public datasets are used as auxiliary priors for geometry, intrinsic appearance, and indoor lighting. A custom hallway dataset is optional but important because public datasets do not directly provide hallway lux labels under fixtures and between fixtures.

Supported datasets in this notebook:
- NYU Depth V2
- MIT Intrinsic Images
- MID Intrinsics
- Fast Spatially-Varying Indoor Lighting Estimation dataset
- optional custom hallway dataset

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print(f'Notebook path: {PROJECT_ROOT / "notebooks" / "hallway_illuminance_train_eval_all_in_one.ipynb"}')

## 2. Dependency Installation

Run this once per Colab session. The requirements include PyTorch, manifest tooling, image tooling, and `.mat` readers used by NYU Depth V2 ingestion.

In [ ]:
requirements_file = PROJECT_ROOT / 'requirements.txt'
print(f'Install dependencies from: {requirements_file}')
# In Colab, uncomment the next line:
# %pip install -q -r ../requirements.txt

## 3. Mounting Google Drive

Google Drive is the intended place for official dataset files, extracted folders, and saved outputs.

In [ ]:
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    print('google.colab is not available in this environment.')

print(f'Running in Colab: {IN_COLAB}')

## 4. Dataset Root Path Configuration

Set each input path to either an extracted folder or an official archive file. For NYU Depth V2, a `.mat` file is supported. For the custom hallway dataset, point to a folder containing a manifest CSV or directly to the CSV itself.

Accepted input types by dataset:
- NYU Depth V2: folder, `.mat`, `.zip`, `.tar`, `.tar.gz`, `.tgz`
- MIT Intrinsic Images: folder, `.zip`, `.tar`, `.tar.gz`, `.tgz`
- MID Intrinsics: folder, `.zip`, `.tar`, `.tar.gz`, `.tgz`
- Fast Indoor Lighting: folder, `.zip`, `.tar`, `.tar.gz`, `.tgz`
- custom hallway: folder containing a CSV manifest, or a direct CSV path

In [ ]:
import pandas as pd

from hallway_lighting.utils.io import ensure_dir, load_yaml
from hallway_lighting.data.dataset_registry import list_supported_datasets

datasets_config = load_yaml(PROJECT_ROOT / 'configs' / 'datasets.yaml')
WORKING_DIR = ensure_dir(PROJECT_ROOT / 'runs' / 'prepared_datasets')
MANIFEST_DIR = ensure_dir(PROJECT_ROOT / 'runs' / 'manifests')

DATASET_INPUTS = {
    'nyu_depth_v2': datasets_config['dataset_roots'].get('nyu_depth_v2', ''),
    'mit_intrinsic_images': datasets_config['dataset_roots'].get('mit_intrinsic_images', ''),
    'mid_intrinsics': datasets_config['dataset_roots'].get('mid_intrinsics', ''),
    'fast_sv_indoor_lighting': datasets_config['dataset_roots'].get('fast_sv_indoor_lighting', ''),
    'custom_hallway': datasets_config['dataset_roots'].get('custom_hallway', ''),
}

# Example Colab edits:
# DATASET_INPUTS['nyu_depth_v2'] = '/content/drive/MyDrive/datasets/nyu_depth_v2/nyu_depth_v2_labeled.mat'
# DATASET_INPUTS['mit_intrinsic_images'] = '/content/drive/MyDrive/datasets/mit_intrinsic_images'
# DATASET_INPUTS['mid_intrinsics'] = '/content/drive/MyDrive/datasets/mid_intrinsics.zip'
# DATASET_INPUTS['fast_sv_indoor_lighting'] = '/content/drive/MyDrive/datasets/fast_sv_indoor_lighting'
# DATASET_INPUTS['custom_hallway'] = '/content/drive/MyDrive/datasets/custom_hallway/custom_hallway_manifest.csv'

display(pd.DataFrame({'dataset_name': list_supported_datasets(), 'input_path': [DATASET_INPUTS[name] for name in list_supported_datasets()]}))
print(f'Working directory: {WORKING_DIR}')
print(f'Manifest directory: {MANIFEST_DIR}')

## 5. Archive Extraction / Dataset Preparation

This step detects each input type, extracts archives into the local working directory under `runs/`, stages `.mat` files safely, and leaves already-extracted folders untouched. It is safe to rerun because extraction is skipped when prepared content already exists and `overwrite=False`.

In [ ]:
from hallway_lighting.data.archive_utils import prepare_dataset_input

PREPARED_INPUTS = {}
prepared_rows = []
for dataset_name, raw_path in DATASET_INPUTS.items():
    if not raw_path:
        continue
    prepared = prepare_dataset_input(
        input_path=raw_path,
        dataset_name=dataset_name,
        working_dir=WORKING_DIR,
        overwrite=False,
    )
    PREPARED_INPUTS[dataset_name] = prepared
    prepared_rows.append(
        {
            'dataset_name': dataset_name,
            'source_path': str(prepared.source_path),
            'input_type': prepared.input_type,
            'prepared_root': str(prepared.prepared_root),
            'primary_file': '' if prepared.primary_file is None else str(prepared.primary_file),
        }
    )

if prepared_rows:
    display(pd.DataFrame(prepared_rows))
else:
    print('Set at least one dataset input path before running preparation.')

## 6. Manifest Generation

This step builds a normalized manifest for every dataset input path provided above. Missing fields are left empty rather than invented. The saved CSVs under `runs/manifests/` are the main bridge between official dataset files and later training code.

In [ ]:
from hallway_lighting.data.dataset_registry import build_all_dataset_manifests
from hallway_lighting.data.manifests import preview_manifest_rows

DATASET_RESULTS = build_all_dataset_manifests(
    dataset_inputs=DATASET_INPUTS,
    working_dir=WORKING_DIR,
    output_dir=MANIFEST_DIR,
    overwrite=False,
)
MANIFESTS = {name: result.manifest for name, result in DATASET_RESULTS.items()}

if not DATASET_RESULTS:
    print('No dataset manifests were built because no input paths were provided.')
else:
    for dataset_name, result in DATASET_RESULTS.items():
        print(f'\n[{dataset_name}] prepared_root={result.prepared_input.prepared_root}')
        print(f'[{dataset_name}] manifest_path={result.manifest_path}')
        display(preview_manifest_rows(result.manifest, limit=5))

## 7. Sample Visualization

This section previews the first visual sample from each manifest where standard image loading is feasible. It helps verify that the official files were located correctly before training.

In [ ]:
import numpy as np
from PIL import Image

from hallway_lighting.data.custom_hallway import load_point_target_values
from hallway_lighting.utils.visualization import show_image

if not DATASET_RESULTS:
    print('Build manifests first.')
else:
    for dataset_name, result in DATASET_RESULTS.items():
        manifest = result.manifest
        if manifest.empty:
            print(f'{dataset_name}: manifest is empty.')
            continue

        first_row = manifest.iloc[0]
        image_path = Path(first_row['image_path'])
        print(f'\nDataset: {dataset_name}')
        print(f'Image path: {image_path}')

        if dataset_name == 'custom_hallway' and isinstance(first_row.get('point_targets_json', ''), str) and first_row.get('point_targets_json', ''):
            print('Point target labels:', load_point_target_values(first_row['point_targets_json']))

        if not image_path.exists():
            print('Image path does not exist on disk.')
            continue
        if image_path.suffix.lower() == '.exr':
            print('EXR preview is skipped in the default notebook preview cell.')
            continue

        try:
            image = np.asarray(Image.open(image_path).convert('RGB'))
            show_image(image, title=f'{dataset_name}: {image_path.name}')
        except Exception as error:
            print(f'Preview failed for {image_path}: {error}')

## 8. Model Setup

The model remains path-driven through YAML configs. The data layer above is now real enough that later cells can consume saved manifests instead of placeholder folder assumptions.

In [ ]:
import torch

from hallway_lighting.models.hallway_multitask_unet import HallwayMultitaskUNet

base_config = load_yaml(PROJECT_ROOT / 'configs' / 'base.yaml')
train_config = load_yaml(PROJECT_ROOT / 'configs' / 'train.yaml')
carbon_config = load_yaml(PROJECT_ROOT / 'configs' / 'carbon.yaml')
infer_config = load_yaml(PROJECT_ROOT / 'configs' / 'infer.yaml')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
run_dir = ensure_dir(PROJECT_ROOT / 'runs' / 'colab_session')
model = HallwayMultitaskUNet(base_config['model']).to(device)

print(model.__class__.__name__)
print(f'Device: {device}')
print(f'Run dir: {run_dir}')

## 9. Training Setup

The dataset preparation phase now writes normalized manifests. The remaining missing piece is the multi-dataset `Dataset` and `DataLoader` implementation that consumes them directly.

In [ ]:
from hallway_lighting.losses.carbon_losses import carbon_interval_loss
from hallway_lighting.losses.intrinsic_losses import intrinsic_reconstruction_loss
from hallway_lighting.losses.lux_losses import lux_map_loss
from hallway_lighting.losses.segmentation_losses import segmentation_loss
from hallway_lighting.losses.uncertainty_losses import heteroscedastic_l1_loss
from hallway_lighting.utils.seed import set_seed

set_seed(base_config['project']['seed'])
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_config['training']['learning_rate'],
    weight_decay=train_config['training']['weight_decay'],
)

# TODO: Replace these placeholders with Dataset/DataLoader objects that read from MANIFESTS.
train_loader = None
val_loader = None
test_loader = None
loss_weights = train_config['loss_weights']
loss_weights

## 10. Training

This loop is ready for real manifests but still waits on the dataset classes that will batch images and whichever labels are available per sample.

In [ ]:
if train_loader is None:
    print('TODO: build train_loader from MANIFESTS before running training.')
else:
    model.train()
    for epoch in range(train_config['training']['epochs']):
        running_loss = 0.0
        for batch in train_loader:
            images = batch['image'].to(device)
            outputs = model(images)

            total_loss = 0.0
            if 'lux_map' in batch:
                target_lux = batch['lux_map'].to(device)
                total_loss = total_loss + loss_weights['lux_map'] * lux_map_loss(outputs['lux_map'], target_lux)
                total_loss = total_loss + loss_weights['uncertainty'] * heteroscedastic_l1_loss(
                    outputs['lux_map'], target_lux, outputs['uncertainty_log_var']
                )
            if 'reflectance' in batch and 'shading' in batch:
                total_loss = total_loss + loss_weights['intrinsic'] * intrinsic_reconstruction_loss(
                    images,
                    outputs['reflectance'],
                    outputs['shading'],
                )
            if 'floor_mask' in batch:
                total_loss = total_loss + loss_weights['floor_mask'] * segmentation_loss(
                    outputs['floor_mask_logits'],
                    batch['floor_mask'].to(device),
                )
            if 'interval_carbon_kg' in batch:
                total_loss = total_loss + loss_weights['carbon'] * carbon_interval_loss(
                    outputs['estimated_power_w'],
                    batch['interval_carbon_kg'].to(device),
                    carbon_config['carbon']['default_grid_carbon_factor_kg_per_kwh'],
                    carbon_config['carbon']['default_interval_hours'],
                )

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), train_config['optimization']['gradient_clip_norm'])
            optimizer.step()
            running_loss += float(total_loss.detach().cpu())

        print(f'Epoch {epoch + 1}: train_loss={running_loss:.4f}')

## 11. Validation

Validation should use the same normalized manifest schema and only compute losses or metrics for labels that actually exist.

In [ ]:
from hallway_lighting.utils.metrics import regression_metrics, summarize_lux_map

if val_loader is None:
    print('TODO: build val_loader before running validation.')
else:
    model.eval()
    validation_rows = []
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(batch['image'].to(device))
            if 'lux_map' in batch:
                target_lux = batch['lux_map'].to(device)
                metrics = regression_metrics(outputs['lux_map'], target_lux)
                metrics.update(summarize_lux_map(outputs['lux_map']))
                validation_rows.append(metrics)
    display(pd.DataFrame(validation_rows)) if validation_rows else print('No validation rows were collected.')

## 12. Testing / Evaluation

Use the held-out manifest split once the dataset loader is connected.

In [ ]:
if test_loader is None:
    print('TODO: build test_loader before running evaluation.')
else:
    model.eval()
    test_rows = []
    with torch.no_grad():
        for batch in test_loader:
            outputs = model(batch['image'].to(device))
            if 'lux_map' in batch:
                test_rows.append(regression_metrics(outputs['lux_map'], batch['lux_map'].to(device)))
    display(pd.DataFrame(test_rows).describe()) if test_rows else print('No test rows were collected.')

## 13. Point-wise Illuminance Reporting

Two related data concepts appear here:
- custom hallway point-target JSON files store measured lux values by name
- inference-time sampling uses normalized point coordinates, which the notebook currently defaults to a simple hallway layout

In [ ]:
from hallway_lighting.data.point_sampling import default_hallway_points, sample_values_at_points
from hallway_lighting.utils.visualization import plot_pointwise_lux

if 'custom_hallway' in MANIFESTS and not MANIFESTS['custom_hallway'].empty:
    custom_manifest = MANIFESTS['custom_hallway']
    point_json_paths = [value for value in custom_manifest['point_targets_json'].fillna('').tolist() if value]
    if point_json_paths:
        print('Example custom hallway point-target labels:')
        print(load_point_target_values(point_json_paths[0]))

example_points = default_hallway_points()
dummy_lux_map = torch.ones(1, 1, 64, 64) * 150.0
point_report = sample_values_at_points(dummy_lux_map, example_points)
display(pd.DataFrame({'point_name': list(point_report.keys()), 'lux': list(point_report.values())}))
plot_pointwise_lux(point_report)

## 14. Carbon Estimation

Carbon is computed from estimated or measured lighting electricity use. The default helper converts illuminance to power, then power to interval energy and carbon.

In [ ]:
from hallway_lighting.utils.carbon import summarize_carbon_from_lux

avg_lux_example = 180.0
carbon_report = summarize_carbon_from_lux(
    avg_lux=avg_lux_example,
    floor_area_m2=infer_config['inference']['floor_area_m2'],
    watts_per_lux_m2=carbon_config['carbon']['watts_per_lux_m2'],
    interval_hours=carbon_config['carbon']['default_interval_hours'],
    carbon_factor_kg_per_kwh=carbon_config['carbon']['default_grid_carbon_factor_kg_per_kwh'],
)
carbon_report

## 15. Checkpoint Saving

All outputs should stay under `runs/` so notebook sessions remain organized.

In [ ]:
from hallway_lighting.utils.io import save_checkpoint

checkpoint_path = run_dir / 'checkpoints' / 'epoch_000.pt'
print(f'Checkpoint will be saved to: {checkpoint_path}')
# save_checkpoint(model, optimizer, epoch=0, path=checkpoint_path, extra_state={'note': 'after dataset preparation'})

## 16. ONNX Export

Export once the checkpointed model is stable enough for deployment experiments.

In [ ]:
onnx_path = PROJECT_ROOT / infer_config['inference']['export_onnx_path']
onnx_path.parent.mkdir(parents=True, exist_ok=True)
dummy_input = torch.randn(
    1,
    3,
    infer_config['inference']['image_size']['height'],
    infer_config['inference']['image_size']['width'],
).to(device)

# torch.onnx.export(
#     model,
#     dummy_input,
#     onnx_path,
#     input_names=['image'],
#     output_names=['lux_map', 'reflectance', 'shading', 'floor_mask_logits', 'uncertainty_log_var', 'estimated_power_w'],
#     opset_version=17,
# )
print(f'Planned ONNX export path: {onnx_path}')

## 17. Single-Image Inference

For inference-time point sampling, the notebook currently uses a default hallway point layout unless you later add a separate coordinate file format.

In [ ]:
from hallway_lighting.data.point_sampling import default_hallway_points
from hallway_lighting.infer import run_single_image_inference

single_image_path = ''
inference_points = default_hallway_points()

if single_image_path and Path(single_image_path).exists():
    inference_output = run_single_image_inference(
        model=model,
        image_path=single_image_path,
        device=device,
        image_size=(
            infer_config['inference']['image_size']['height'],
            infer_config['inference']['image_size']['width'],
        ),
        point_targets=inference_points,
        carbon_factor_kg_per_kwh=carbon_config['carbon']['default_grid_carbon_factor_kg_per_kwh'],
        interval_hours=infer_config['inference']['interval_hours'],
    )
    print(inference_output)
else:
    print('Set single_image_path to a real hallway image before running inference.')

## 18. Next Steps

The data layer is now path-driven and notebook-usable. The next implementation phase should focus on:
- building multi-dataset `Dataset` and `DataLoader` classes that read the normalized manifests
- handling missing labels cleanly across datasets
- defining training splits when a source dataset does not ship with one in an easy text-file format
- connecting real batch dictionaries to the training, validation, and evaluation cells
- adding hallway-specific supervision workflows for lux maps, point targets, measured power, and calibrated carbon reporting